In [44]:
!pip install -q langchain langgraph langchain-community langchain-chroma chromadb
!pip install -q sentence-transformers duckduckgo-search groq

In [45]:
!pip install -q langchain-groq

In [46]:
!rm -rf chroma_db

In [47]:
import os
from getpass import getpass

os.environ["GROQ_API_KEY"] = getpass("Enter Groq API Key: ")

Enter Groq API Key: ··········


In [48]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0
)

In [49]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [50]:
from langchain_community.document_loaders import TextLoader

file_path = "/content/software_manual.txt"

loader = TextLoader(file_path, encoding="utf-8")
docs = loader.load()

print(docs[0].page_content[:400])

﻿======================================================================
  ENTERPRISE TECHNICAL MANUAL & FAQ: ORION SMARTHUB X1 PRO GATEWAY
Product Engine: Orion Core Ecosystem
Hardware Model: SmartHub-X1-PRO
Firmware Baseline: Stable Build v4.12.0
Document ID: TECH-REF-2026-REV2


1. HARDWARE SPECIFICATIONS & INTERNAL CHIPSETS



In [51]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50
)

chunks = splitter.split_documents(docs)

print("Chunks:", len(chunks))

Chunks: 22


In [53]:
from langchain_chroma import Chroma
persist_directory = "/content/chroma_db"
os.makedirs(persist_directory, exist_ok=True)
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=persist_directory
)

retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

In [54]:
query = "how do I hard factory reset?"

results = retriever.invoke(query)

for i, doc in enumerate(results):
    print(f"\n--- RESULT {i+1} ---")
    print(doc.page_content)


--- RESULT 1 ---
4. SYSTEM RESTORATION & HARD RESET ARCHITECTURE
----------------------------------------------------------------------
- Soft System Reboot: Log into the cloud dashboard panel, navigate to 
  System Admin Console > Maintenance Tools, and commit the "Restart Hub"

--- RESULT 2 ---
pipeline. Hardware method: Disconnect the power coupling from the 
  USB-C terminal port for exactly 15 seconds, then re-insert.
  
- Hard Factory Reset: Locate the recessed sub-surface microswitch button 
  labeled "RESET" on the rear panel assembly (situated directly to the

--- RESULT 3 ---
Step 1: Inspect the hardware for integrity, connect the OEM USB-C 
        power supply, and connect to a power outlet.
Step 2: Monitor the chassis faceplate. Wait for the primary LED to 
        blink in an Amber color pattern, signaling "Ready for Provisioning."


In [55]:
def rewrite_query(state):
    query = state["original_query"]

    prompt = f"""
Convert this into a short keyword search query:

{query}

Return ONLY keywords.
"""

    response = llm.invoke(prompt)

    return {
        "optimized_query": response.content.strip()
    }

In [56]:
state = {
    "original_query": "How do i hard factory reset",
    "optimized_query": "",
    "documents": [],
    "generation": "",
    "loop_count": 0
}

print(rewrite_query(state))

{'optimized_query': '"hard factory reset"'}


In [57]:
from duckduckgo_search import DDGS

def web_search(query):
    results = []

    with DDGS() as ddgs:
        for r in ddgs.text(query, max_results=3):
            results.append(r["body"])

    return results

In [58]:
def grade_documents(state):
    query = state["optimized_query"]
    docs = retriever.invoke(query)

    prompt = f"""
You are a strict QA engineer.

Task:
Check if the documents are relevant to the query.

Query: {query}

Documents:
{[d.page_content for d in docs]}

Return ONLY JSON:
{{
  "relevance_score": "yes" or "no"
}}
"""

    response = llm.invoke(prompt)

    try:
        decision = response.content.lower()
        is_relevant = "yes" in decision
    except:
        is_relevant = False

    return {
        "documents": [d.page_content for d in docs],
        "relevant": is_relevant
    }

In [59]:
def route_decision(state):
    if state["relevant"]:
        return "generate"
    else:
        return "web_search"

In [60]:
def web_search_node(state):
    query = state["optimized_query"]

    results = web_search(query)

    return {
        "documents": results,
        "web_search": True
    }

In [61]:
def generate_answer(state):
    context = "\n\n".join(state["documents"])
    query = state["optimized_query"]

    prompt = f"""
You are a technical support assistant.

Use ONLY the context below to answer.

Context:
{context}

Question:
{query}

Rules:
- Do NOT hallucinate
- Be precise
- If unsure say "Not found in documentation"
"""

    response = llm.invoke(prompt)

    return {
        "generation": response.content
    }

In [62]:
from typing import TypedDict, List

class AgentState(TypedDict):
    original_query: str
    optimized_query: str
    documents: List[str]
    generation: str
    loop_count: int
    relevant: bool
    web_search: bool
    hallucination: str

In [63]:
def hallucination_check(state):
    context = "\n".join(state["documents"])
    answer = state["generation"]

    prompt = f"""
You are a strict QA engineer.

Check if the answer is fully supported by the context.

Context:
{context}

Answer:
{answer}

Return ONLY JSON:
{{
  "hallucination_check": "passed" or "failed"
}}
"""

    response = llm.invoke(prompt)

    text = response.content.lower()

    if "passed" in text:
        return {"hallucination": "passed"}
    else:
        return {"hallucination": "failed"}

In [64]:
def check_loop(state):
    if state["loop_count"] >= 2:
        return "end"

    if state["hallucination"] == "passed":
        return "end"

    return "rewrite"

In [65]:
def rewrite_query(state):
    query = state["original_query"]

    prompt = f"""
Improve this search query for better retrieval:

{query}

Make it more specific and technical.
Return ONLY query.
"""

    response = llm.invoke(prompt)

    return {
        "optimized_query": response.content.strip(),
        "loop_count": state["loop_count"] + 1
    }

In [66]:
def generate_answer(state):
    context = "\n\n".join(state["documents"])
    query = state["optimized_query"]

    prompt = f"""
You are a strict enterprise support assistant.

Use ONLY the context below.

Context:
{context}

Question:
{query}

Rules:
- Do NOT guess
- Do NOT hallucinate
- If missing say "Not found in documentation"
"""

    response = llm.invoke(prompt)

    return {
        "generation": response.content
    }

In [67]:
from langgraph.graph import StateGraph, END

In [68]:
workflow = StateGraph(AgentState)

In [69]:
workflow.add_node("rewrite", rewrite_query)
workflow.add_node("retrieve", lambda state: state)
workflow.add_node("web_search", web_search_node)
workflow.add_node("generate", generate_answer)
workflow.add_node("hallucination", hallucination_check)

workflow.set_entry_point("rewrite")

workflow.add_edge("rewrite", "retrieve")
workflow.add_edge("retrieve", "generate")
workflow.add_edge("web_search", "generate")
workflow.add_edge("generate", "hallucination")

In [70]:
def route_retrieval(state):
    return "retrieve"

In [71]:
def route_after_generate(state):
    if state["hallucination"] == "passed":
        return "__end__"

    if state["loop_count"] >= 2:
        return "__end__"

    return "rewrite"

In [72]:
workflow.add_conditional_edges(
    "hallucination",
    route_after_generate,
    {
        "rewrite": "rewrite",
        "__end__": END
    }
)

In [73]:
app = workflow.compile()

print("LangGraph system compiled successfully.")

LangGraph system compiled successfully.


In [74]:
initial_state = {
    "original_query": "how do I hard factory rest?",
    "optimized_query": "",
    "documents": [],
    "generation": "",
    "loop_count": 0,
    "relevant": False,
    "web_search": False,
    "hallucination": ""
}

result = app.invoke(initial_state)

print(result["generation"])

To provide accurate information, I'll need to refer to the documentation for the factory reset process of hardware devices.

For a factory reset, the steps may vary depending on the device type and manufacturer. Here are some general steps for common devices:

1. **Desktop Computers:**
   - Not found in documentation

2. **Laptops:**
   - Not found in documentation

3. **Smartphones:**
   - Not found in documentation

4. **Tablets:**
   - Not found in documentation

5. **Routers:**
   - Not found in documentation

6. **Printers:**
   - Not found in documentation

7. **Smart Home Devices:**
   - Not found in documentation

8. **Networking Devices:**
   - Not found in documentation

9. **Servers:**
   - Not found in documentation

10. **Other Devices:**
    - Not found in documentation

Please provide the specific device type and model for more accurate information.
